# **Libraries**

In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model, Environment, ManagedOnlineEndpoint, ManagedOnlineDeployment, CodeConfiguration
from azure.identity import DefaultAzureCredential
import json
import os
import requests

# **Get ML Client**

In [ ]:
ml_client = MLClient.from_config(credential= DefaultAzureCredential())

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [ ]:
ml_client

MLClient(credential=<azure.identity._credentials.default.DefaultAzureCredential object at 0x7eb352bb9750>,
         subscription_id=cb199ddd-a44a-4bd8-9e03-3a31418f30d0,
         resource_group_name=rg-pure-prem,
         workspace_name=purprem_ws)

# **Register Model and Environment**

In [ ]:
%cat env.yml

name: lgbm-env
channels: [conda-forge]
dependencies:
  - python=3.10
  - pip
  - pip:
      - lightgbm==4.5.0
      - scikit-learn==1.5
      - scipy==1.13
      - pandas==2.2
      - numpy==1.26
      - pydantic
      - azureml-inference-server-http

In [ ]:
model = ml_client.models.create_or_update(
    Model(path = 'loss_cost_lgbm.txt', name = 'Loss_Cost_LGBM', type='custom_model')
)
env = ml_client.environments.create_or_update(
    Environment(
        name = 'Loss_Cost_LGBM_Env',
        conda_file = 'env.yml',
        image = 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04'
    )
)

# **Register Endpoint**

In [ ]:
endpoint = ManagedOnlineEndpoint(
    name = 'loss-cost-lgbm-endpoint-v4',
    auth_mode = 'key'
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

ManagedOnlineEndpoint({'public_network_access': <PublicNetworkAccessType.ENABLED: 'Enabled'>, 'provisioning_state': <EndpointProvisioningState.SUCCEEDED: 'Succeeded'>, 'scoring_uri': 'https://loss-cost-lgbm-endpoint-v4.canadacentral.inference.ml.azure.com/score', 'openapi_uri': 'https://loss-cost-lgbm-endpoint-v4.canadacentral.inference.ml.azure.com/swagger.json', 'name': 'loss-cost-lgbm-endpoint-v4', 'description': None, 'tags': {}, 'properties': {'createdBy': 'Mezmur Edo', 'createdAt': '2026-11-02T18:07:11.673020+0000', 'lastModifiedAt': '2026-11-02T18:08:33.955150+0000', 'azureml.onlineendpointid': '/subscriptions/cb199ddd-a44a-4bd8-9e03-3a31418f30d0/resourcegroups/rg-pure-prem/providers/microsoft.machinelearningservices/workspaces/purprem_ws/onlineendpoints/loss-cost-lgbm-endpoint-v4', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/cb199ddd-a44a-4bd8-9e03-3a31418f30d0/providers/Microsoft.MachineLearningServices/locations/canadacentral/mfeOperationsStatus/oeid

# **Deploy**

In [ ]:
%cat predict.py

"""predict.py
"""
import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import pandas as pd
import lightgbm as lgb
from functools import lru_cache
from features import build_features
from config import Config
import json

def init():
    global model
    model = lgb.Booster(model_file = 'loss_cost_lgbm.txt')


def run(raw_data):
    policy_df = pd.DataFrame([json.loads(raw_data)])
    feature_df = build_features(policy_df)
    loss_cost = model.predict(feature_df)
    return {'loss_cost': float(loss_cost[0])}

In [ ]:
deployment = ManagedOnlineDeployment(
    name = 'blue',
    endpoint_name= 'loss-cost-lgbm-endpoint-v4',
    model = model,
    environment = env,
    code_configuration = CodeConfiguration(code  = '.', scoring_script = 'predict.py'),
    instance_type = 'Standard_DS2_v2',
    instance_count = 1
)

ml_client.online_deployments.begin_create_or_update(deployment).result()

Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint loss-cost-lgbm-endpoint-v4 exists
Uploading lgbm (0.11 MBs): 100%|██████████| 107603/107603 [00:00<00:00, 705448.26it/s]




...........................................

ManagedOnlineDeployment({'private_network_connection': None, 'package_model': False, 'provisioning_state': <DeploymentProvisioningState.SUCCEEDED: 'Succeeded'>, 'endpoint_name': 'loss-cost-lgbm-endpoint-v4', 'type': 'Managed', 'name': 'blue', 'description': None, 'tags': {}, 'properties': {'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/cb199ddd-a44a-4bd8-9e03-3a31418f30d0/providers/Microsoft.MachineLearningServices/locations/canadacentral/mfeOperationsStatus/odidp:cf8a424c-550c-4760-8e2d-75cd6579165d:0c77f3a0-7a1b-4af4-867a-3e96c9d248db?api-version=2023-04-01-preview'}, 'print_as_yaml': False, 'id': '/subscriptions/cb199ddd-a44a-4bd8-9e03-3a31418f30d0/resourceGroups/rg-pure-prem/providers/Microsoft.MachineLearningServices/workspaces/purprem_ws/onlineEndpoints/loss-cost-lgbm-endpoint-v4/deployments/blue', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/azure-ml-loss-cost/code/Users/mezmurw.edo/lgbm', 'creation_context': {

# **Usage via Requests**

In [ ]:
endpoint = ml_client.online_endpoints.get('loss-cost-lgbm-endpoint-v4')
scoring_uri = endpoint.scoring_uri
keys = ml_client.online_endpoints.get_keys('loss-cost-lgbm-endpoint-v4')
key = keys.primary_key

In [ ]:
payload = {
    "RiskArea": 5.0,
    "BonusMalus": 100,
    "DrivAge": 35,
    "LicAge": 120,
    "VehUsage": "Private"
}

headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {key}',
    'azureml-model-deployment': 'blue'
}

response = requests.post(
    scoring_uri,
    headers = headers,
    json = payload,
)

result = response.json()
print(result)

{'loss_cost': 570.4891651846364}


In [ ]:
ml_client.online_endpoints.begin_delete(name="loss-cost-lgbm-endpoint-v2").result()
print("Successfully deleted endpoint.")

ml_client.online_deployments.begin_delete(
    name='blue',
    endpoint_name='loss-cost-lgbm-endpoint-v4'
).result()
print("Successfully deleted deployment.")

Successfully deleted endpoint.
Successfully deleted deployment.
